<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/dnn-course-2026-1/blob/main/C1_M1_Lab_3_tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Tensores: O "Núcleo" do PyTorch

Você viu que a jornada de construção de uma rede neural começa com os dados. Antes de poder projetar um modelo ou iniciar o processo de treinamento, você deve reunir suas informações e prepará-las em um formato que o modelo consiga entender. No PyTorch, esse formato fundamental é o **tensor**. Tensores são mais do que apenas recipientes de dados; eles são otimizados para as operações matemáticas que impulsionam o *deep learning*.

Dominar os tensores é uma etapa vital. Muitos dos erros mais comuns encontrados ao construir modelos estão relacionados a formatos (*shapes*), tipos ou dimensões dos tensores. Este laboratório foi projetado para fornecer uma base sólida em manipulação de tensores, oferecendo as habilidades necessárias para lidar com dados de forma eficaz e depurar problemas com confiança.

Neste laboratório, você aprenderá como:

* Criar tensores a partir de diferentes fontes de dados, como listas Python e matrizes NumPy.

* Remodelar (*reshape*) e manipular dimensões de tensores para preparar os dados para as entradas do modelo.

* Usar técnicas de indexação e fatiamento (*slicing*) para acessar e filtrar partes específicas dos seus dados.

* Realizar as operações matemáticas e lógicas que formam a base de todos os cálculos de redes neurais.

Ao final deste notebook, você terá as habilidades práticas necessárias para gerenciar com confiança os dados de qualquer projeto em PyTorch.

## Importar bibliotecas

In [1]:
import os
# Verificar se o arquivo existe localmente antes de baixar
if not os.path.exists('helper_utils_1.py'):
    !wget -q https://raw.githubusercontent.com/fabiobento/dnn-course-2026-1/main/data.csv -O data.csv

In [3]:
import torch
import numpy as np
import pandas as pd

## 1 - Criação de Tensores

O primeiro passo em qualquer pipeline de machine learning é deixar seus dados prontos para o modelo. No PyTorch, isso significa carregar seus dados em tensores. Você descobrirá que existem várias maneiras convenientes de criar tensores, esteja seu dado já em outro formato ou se você precisar gerá-lo do zero.

### 1.1 A partir de Estruturas de Dados Existentes

Frequentemente, seus dados brutos estarão em um formato comum, como uma lista Python, um array NumPy ou um DataFrame do pandas. O PyTorch fornece funções diretas para converter essas estruturas em tensores, tornando a etapa de preparação de dados mais eficiente.

* `torch.tensor()`: Esta função recebe uma entrada, como uma lista Python, para convertê-la em um tensor.

**Nota:** O tipo de números que você usa é importante. Se você usar números inteiros, o PyTorch os armazenará como inteiros. Se incluir decimais, eles serão armazenados como valores de ponto flutuante (*floating point*).

In [4]:
# A partir de listas Python
x = torch.tensor([1, 2, 3])

print("DE LISTAS PYTHON:", x)
print("TIPO DE DADOS DO TENSOR:", x.dtype)

DE LISTAS PYTHON: tensor([1, 2, 3])
TIPO DE DADOS DO TENSOR: torch.int64


* `torch.from_numpy()`: Converte um array NumPy em um tensor PyTorch.

In [5]:
# A partir de um array NumPy
numpy_array = np.array([[1, 2, 3], [4, 5, 6]])
torch_tensor_from_numpy = torch.from_numpy(numpy_array)

print("TENSOR A PARTIR DO NUMPY:\n\n", torch_tensor_from_numpy)

TENSOR A PARTIR DO NUMPY:

 tensor([[1, 2, 3],
        [4, 5, 6]])


<br>

* **A partir de um DataFrame do pandas**: O Pandas é uma biblioteca Python para trabalhar com dados organizados em linhas e colunas, como um arquivo CSV ou uma planilha. Um DataFrame é a principal estrutura de dados do pandas para armazenar esse tipo de dado tabular. DataFrames são uma das formas mais comuns de carregar e explorar conjuntos de dados em machine learning, especialmente ao ler arquivos CSV. Não existe uma função direta para converter um DataFrame em um tensor. O método padrão é extrair os dados do DataFrame para um array NumPy usando o atributo `.values` e, em seguida, converter esse array em um tensor usando `torch.tensor()`.

In [7]:
# A partir de um DataFrame do Pandas
# Leia os dados do arquivo CSV para um DataFrame
df = pd.read_csv('./data.csv')

# Extraia os dados do DataFrame como um array NumPy
all_values = df.values

# Converta os valores do DataFrame para um tensor PyTorch
tensor_from_df = torch.tensor(all_values)

print("DATAFRAME ORIGINAL:\n\n", df)
print("\nTENSOR RESULTANTE:\n\n", tensor_from_df)
print("\nTIPO DE DADO DO TENSOR:", tensor_from_df.dtype)

DATAFRAME ORIGINAL:

    distance_miles  delivery_time_minutes
0            1.60                   7.22
1           13.09                  32.41
2            6.97                  17.47

TENSOR RESULTANTE:

 tensor([[ 1.6000,  7.2200],
        [13.0900, 32.4100],
        [ 6.9700, 17.4700]], dtype=torch.float64)

TIPO DE DADO DO TENSOR: torch.float64


### 1.2 - Com Valores Predefinidos

Às vezes, você precisa criar tensores para propósitos específicos, como inicializar os pesos e vieses (*weights* e *biases*) de um modelo antes do início do treinamento. O PyTorch permite gerar rapidamente tensores preenchidos com valores de marcação (*placeholders*), como zeros, uns ou números aleatórios, o que é útil para testes e configurações iniciais.

* `torch.zeros()`: Cria um tensor preenchido com zeros nas dimensões especificadas.

In [8]:
# Tudo zeros
zeros = torch.zeros(2, 3)

print("TENSOR COM ZEROS:\n\n", zeros)

TENSOR COM ZEROS:

 tensor([[0., 0., 0.],
        [0., 0., 0.]])


* `torch.ones()`: Cria um tensor preenchido com o número um nas dimensões especificadas.

In [9]:
# Tudo uns
ones = torch.ones(2, 3)

print("TENSOR COM UNS:\n\n", ones)

TENSOR COM UNS:

 tensor([[1., 1., 1.],
        [1., 1., 1.]])


* `torch.rand()`: Gera um tensor com números aleatórios uniformemente distribuídos entre 0 e 1, com base nas dimensões especificadas.

In [10]:
# Números aleatórios
random = torch.rand(2, 3)

print("TENSOR ALEATÓRIO:\n\n", random)

TENSOR ALEATÓRIO:

 tensor([[0.9973, 0.6186, 0.3522],
        [0.3621, 0.9671, 0.8264]])


### 1.3 - A partir de uma Sequência

Para situações em que você precisa gerar uma sequência de pontos de dados, como um intervalo de valores para testar as previsões de um modelo, você pode criar um tensor diretamente a partir dessa sequência.

* `torch.arange()`: Cria um tensor 1D contendo um intervalo de números, desde o valor inicial especificado até um valor a menos que o valor final (stop), incrementando (se positivo) ou decrementando (se negativo) pelo valor de passo (`step`) especificado.

In [ ]:
# Range of numbers
range_tensor = torch.arange(0, 10, step=1)

print("ARANGE TENSOR:", range_tensor)

## 2 - Redimensionamento e Manipulação

Uma fonte muito comum de erros em projetos de PyTorch é a incompatibilidade entre o formato (*shape*) dos seus dados de entrada e o formato que o seu modelo espera. Por exemplo, um modelo é tipicamente projetado para processar um lote (*batch*) de dados; portanto, mesmo que você queira fazer uma única previsão, deve moldar seu tensor de entrada para que ele pareça um lote de uma única unidade. Dominar o redimensionamento de tensores é um passo fundamental para construir e depurar modelos de forma eficaz.

### 2.1 - Verificando as Dimensões de um Tensor

O primeiro passo para corrigir uma incompatibilidade de formato é entender as dimensões atuais do seu tensor. Verificar o *shape* é sua principal ferramenta de depuração. Ele informa quantos exemplos você possui e quantas características (*features*) existem em cada exemplo.

* `torch.Tensor.shape`: Um atributo que retorna um objeto `torch.Size` detalhando o tamanho do tensor em cada dimensão.

In [11]:
# Um tensor 2D
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

print("TENSOR ORIGINAL:\n\n", x)
print("\nFORMATO DO TENSOR:", x.shape)

TENSOR ORIGINAL:

 tensor([[1, 2, 3],
        [4, 5, 6]])

FORMATO DO TENSOR: torch.Size([2, 3])


### 2.2 - Alterando as Dimensões de um Tensor

Depois de identificar uma incompatibilidade de formato (*shape*), você precisa corrigi-la. Uma tarefa frequente é adicionar uma dimensão a uma única amostra de dados para criar um lote (*batch*) de tamanho um para o seu modelo, ou remover uma dimensão após a conclusão de uma operação de lote.

* **Adicionando Dimensão:** `torch.Tensor.unsqueeze()` insere uma nova dimensão no índice especificado.
    * *Observe como o formato mudará de `[2, 3]` para `[1, 2, 3]` e o tensor ganha um par extra de colchetes `[]` envolvendo-o.*

In [12]:
print("TENSOR ORIGINAL:\n\n", x)
print("\nFORMATO (SHAPE) DO TENSOR:", x.shape)
print("-"*45)

# Adicionar dimensão
expanded = x.unsqueeze(0)  # Adiciona uma dimensão no índice 0

print("\nTENSOR COM DIMENSÃO ADICIONADA NO ÍNDICE 0:\n\n", expanded)
print("\nFORMATO (SHAPE) DO TENSOR:", expanded.shape)

TENSOR ORIGINAL:

 tensor([[1, 2, 3],
        [4, 5, 6]])

FORMATO (SHAPE) DO TENSOR: torch.Size([2, 3])
---------------------------------------------

TENSOR COM DIMENSÃO ADICIONADA NO ÍNDICE 0:

 tensor([[[1, 2, 3],
         [4, 5, 6]]])

FORMATO (SHAPE) DO TENSOR: torch.Size([1, 2, 3])


* **Removendo Dimensão:** `torch.Tensor.squeeze()` remove dimensões de tamanho 1.
    * *Isso reverte a operação de unsqueeze, removendo o `1` do formato (shape) e eliminando um par de colchetes externos.*

In [13]:
print("TENSOR EXPANDIDO:\n\n", expanded)
print("\nFORMATO (SHAPE) DO TENSOR:", expanded.shape)
print("-"*45)

# Remover dimensão
squeezed = expanded.squeeze()

print("\nTENSOR COM DIMENSÃO REMOVIDA:\n\n", squeezed)
print("\nFORMATO (SHAPE) DO TENSOR:", squeezed.shape)

TENSOR EXPANDIDO:

 tensor([[[1, 2, 3],
         [4, 5, 6]]])

FORMATO (SHAPE) DO TENSOR: torch.Size([1, 2, 3])
---------------------------------------------

TENSOR COM DIMENSÃO REMOVIDA:

 tensor([[1, 2, 3],
        [4, 5, 6]])

FORMATO (SHAPE) DO TENSOR: torch.Size([2, 3])


### 2.3 - Reestruturação

Além de apenas adicionar ou remover dimensões, você pode precisar alterar completamente a estrutura de um tensor para atender aos requisitos de uma camada ou operação específica dentro da sua rede neural.

* **Redimensionamento (*Reshaping*):** `torch.Tensor.reshape()` altera o formato de um tensor para as dimensões especificadas.

In [14]:
print("TENSOR ORIGINAL:\n\n", x)
print("\nFORMATO (SHAPE) DO TENSOR:", x.shape)
print("-"*45)

# Alterar o formato (Reshape)
reshaped = x.reshape(3, 2)

print("\nAPÓS EXECUTAR O reshape(3, 2):\n\n", reshaped)
print("\nFORMATO (SHAPE) DO TENSOR:", reshaped.shape)

TENSOR ORIGINAL:

 tensor([[1, 2, 3],
        [4, 5, 6]])

FORMATO (SHAPE) DO TENSOR: torch.Size([2, 3])
---------------------------------------------

APÓS EXECUTAR O reshape(3, 2):

 tensor([[1, 2],
        [3, 4],
        [5, 6]])

FORMATO (SHAPE) DO TENSOR: torch.Size([3, 2])


* **Transpondo:** `torch.Tensor.transpose()` troca as dimensões especificadas de um tensor.

In [15]:
print("TENSOR ORIGINAL:\n\n", x)
print("\nFORMATO (SHAPE) DO TENSOR:", x.shape)
print("-"*45)

# Transpor
transposed = x.transpose(0, 1)

print("\nAPÓS EXECUTAR O transpose(0, 1):\n\n", transposed)
print("\nFORMATO (SHAPE) DO TENSOR:", transposed.shape)

TENSOR ORIGINAL:

 tensor([[1, 2, 3],
        [4, 5, 6]])

FORMATO (SHAPE) DO TENSOR: torch.Size([2, 3])
---------------------------------------------

APÓS EXECUTAR O transpose(0, 1):

 tensor([[1, 4],
        [2, 5],
        [3, 6]])

FORMATO (SHAPE) DO TENSOR: torch.Size([3, 2])


### 2.4 - Combinando Tensores

Na etapa de preparação de dados, você pode precisar combinar dados de diferentes fontes ou mesclar lotes (*batches*) separados em um único conjunto de dados maior.

* `torch.cat()`: Une uma sequência de tensores ao longo de uma dimensão existente.

> **Nota**:
> 
> Todos os tensores devem ter o mesmo formato nas dimensões que não sejam aquela onde a concatenação está ocorrendo.

In [16]:
# Criar dois tensores para concatenar
tensor_a = torch.tensor([[1, 2],
                         [3, 4]])
tensor_b = torch.tensor([[5, 6],
                         [7, 8]])

# Concatenar ao longo das colunas (dim=1)
concatenated_tensors = torch.cat((tensor_a, tensor_b), dim=1)


print("TENSOR A:\n\n", tensor_a)
print("\nTENSOR B:\n\n", tensor_b)
print("-"*45)
print("\nTENSOR CONCATENADO (dim=1):\n\n", concatenated_tensors)

TENSOR A:

 tensor([[1, 2],
        [3, 4]])

TENSOR B:

 tensor([[5, 6],
        [7, 8]])
---------------------------------------------

TENSOR CONCATENADO (dim=1):

 tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])


## 3 - Indexação e Fatiamento

Depois que os seus dados estão em um tensor, você frequentemente precisará acessar partes específicas deles. Seja para capturar uma única previsão e inspecionar seu valor, separar as características de entrada (*features*) dos seus rótulos (*labels*), ou selecionar um subconjunto de dados para análise, a indexação e o fatiamento (*slicing*) são as ferramentas para essa tarefa.

### 3.1 - Acessando Elementos

Estas são as técnicas fundamentais para extrair dados de um tensor, funcionando de forma muito semelhante à maneira como você acessaria elementos em uma lista padrão do Python.

* **Indexação Padrão**: Acessar elementos individuais ou linhas inteiras usando índices inteiros (ex: `x[0]`, `x[1, 2]`).

In [17]:
# Criar um tensor 3x4
x = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])
print("TENSOR ORIGINAL:\n\n", x)
print("-" * 55)

# Obter um único elemento na linha 1, coluna 2
single_element_tensor = x[1, 2]

print("\nINDEXANDO ELEMENTO ÚNICO EM [1, 2]:", single_element_tensor)
print("-" * 55)

# Obter toda a segunda linha (índice 1)
second_row = x[1]

print("\nINDEXANDO LINHA INTEIRA [1]:", second_row)
print("-" * 55)

# Última linha
last_row = x[-1]

print("\nINDEXANDO TODA A ÚLTIMA LINHA ([-1]):", last_row, "\n")

TENSOR ORIGINAL:

 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
-------------------------------------------------------

INDEXANDO ELEMENTO ÚNICO EM [1, 2]: tensor(7)
-------------------------------------------------------

INDEXANDO LINHA INTEIRA [1]: tensor([5, 6, 7, 8])
-------------------------------------------------------

INDEXANDO TODA A ÚLTIMA LINHA ([-1]): tensor([ 9, 10, 11, 12]) 



* **Fatiamento (*Slicing*)**: Extração de sub-tensores usando a notação `[início:fim:passo]` (ex: `x[:2, ::2]`).
    * *Nota: O índice de `fim` (stop) não é incluído no fatiamento.*
* O fatiamento pode ser usado para acessar colunas inteiras.

In [18]:
print("TENSOR ORIGINAL:\n\n", x)
print("-" * 55)

# Obter as duas primeiras linhas
first_two_rows = x[0:2]

print("\nFATIANDO AS DUAS PRIMEIRAS LINHAS ([0:2]):\n\n", first_two_rows)
print("-" * 55)

# Obter a terceira coluna de todas as linhas
third_column = x[:, 2]

print("\nFATIANDO A TERCEIRA COLUNA ([:, 2]]):", third_column)
print("-" * 55)

# Uma coluna sim, outra não (pular colunas)
every_other_col = x[:, ::2]

print("\nUMA COLUNA SIM, OUTRA NÃO ([:, ::2]):\n\n", every_other_col)
print("-" * 55)

# Última coluna
last_col = x[:, -1]

print("\nÚLTIMA COLUNA ([:, -1]):", last_col, "\n")

TENSOR ORIGINAL:

 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
-------------------------------------------------------

FATIANDO AS DUAS PRIMEIRAS LINHAS ([0:2]):

 tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])
-------------------------------------------------------

FATIANDO A TERCEIRA COLUNA ([:, 2]]): tensor([ 3,  7, 11])
-------------------------------------------------------

UMA COLUNA SIM, OUTRA NÃO ([:, ::2]):

 tensor([[ 1,  3],
        [ 5,  7],
        [ 9, 11]])
-------------------------------------------------------

ÚLTIMA COLUNA ([:, -1]): tensor([ 4,  8, 12]) 



* Combinando Indexação e Fatiamento

In [19]:
print("TENSOR ORIGINAL:\n\n", x)
print("-" * 55)

# Combinando fatiamento e indexação (Primeiras duas linhas, últimas duas colunas)
combined = x[0:2, 2:]

print("\nPRIMEIRAS DUAS LINHAS, ÚLTIMAS DUAS COLUNAS ([0:2, 2:]):\n\n", combined, "\n")

TENSOR ORIGINAL:

 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
-------------------------------------------------------

PRIMEIRAS DUAS LINHAS, ÚLTIMAS DUAS COLUNAS ([0:2, 2:]):

 tensor([[3, 4],
        [7, 8]]) 



* **`.item()`**: Extrai o valor de um tensor de um único elemento como um número padrão do Python.

In [20]:
print("TENSOR DE ELEMENTO ÚNICO:", single_element_tensor)
print("-" * 45)

# Extrair o valor de um tensor de elemento único como um número padrão do Python
value = single_element_tensor.item()

print("\nNÚMERO PYTHON EXTRAÍDO COM .item():", value)
print("TIPO:", type(value))

TENSOR DE ELEMENTO ÚNICO: tensor(7)
---------------------------------------------

NÚMERO PYTHON EXTRAÍDO COM .item(): 7
TIPO: <class 'int'>


### 3.2 - Indexação Avançada

Para seleções de dados mais complexas, como filtrar seu conjunto de dados com base em uma ou mais condições, você pode usar técnicas de indexação avançada.

* **Máscara Booleana (Boolean Masking)**: Usar um tensor booleano para selecionar elementos que atendam a uma determinada condição (ex: `x[x > 5]`).

In [21]:
print("TENSOR ORIGINAL:\n\n", x)
print("-" * 55)

# Indexação booleana usando comparações lógicas
mask = x > 6

print("MÁSCARA (VALORES > 6):\n\n", mask, "\n")

# Aplicando a máscara booleana
mask_applied = x[mask]

print("VALORES APÓS APLICAR A MÁSCARA:", mask_applied, "\n")

TENSOR ORIGINAL:

 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
-------------------------------------------------------
MÁSCARA (VALORES > 6):

 tensor([[False, False, False, False],
        [False, False,  True,  True],
        [ True,  True,  True,  True]]) 

VALORES APÓS APLICAR A MÁSCARA: tensor([ 7,  8,  9, 10, 11, 12]) 



* **Indexação Avançada (*Fancy Indexing*)**: Usar um tensor de índices para selecionar elementos específicos de forma não contígua.

In [22]:
print("TENSOR ORIGINAL:\n\n", x)
print("-" * 55)

# Indexação avançada (Fancy indexing)

# Obter a primeira e a terceira linhas
row_indices = torch.tensor([0, 2])

# Obter a segunda e a quarta colunas
col_indices = torch.tensor([1, 3]) 

# Obtém os valores nas posições (0,1), (0,3), (2,1), (2,3)
get_values = x[row_indices[:, None], col_indices]

print("\nELEMENTOS ESPECÍFICOS USANDO ÍNDICES:\n\n", get_values, "\n")

TENSOR ORIGINAL:

 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
-------------------------------------------------------

ELEMENTOS ESPECÍFICOS USANDO ÍNDICES:

 tensor([[ 2,  4],
        [10, 12]]) 



## 4 - Operações Matemáticas e Lógicas

Em sua essência, as redes neurais realizam computações matemáticas. Um único neurônio, por exemplo, calcula uma soma ponderada de suas entradas e adiciona um viés (*bias*). O PyTorch é otimizado para realizar essas operações com eficiência em tensores inteiros de uma só vez, o que torna o treinamento tão rápido.

### 4.1 - Aritmética

Essas operações são a base de como uma rede neural processa dados. Você verá como o PyTorch lida com cálculos elemento a elemento e utiliza um recurso poderoso chamado *broadcasting* para simplificar seu código.

* **Operações Elemento a Elemento (Element-wise)**: Operadores matemáticos padrão (`+`, `*`) que se aplicam a cada elemento de forma independente.

In [23]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
print("TENSOR A:", a)
print("TENSOR B:", b)
print("-" * 60)

# Adição elemento a elemento (element-wise)
element_add = a + b

print("\nAPÓS EXECUTAR A ADIÇÃO ELEMENTO A ELEMENTO:", element_add, "\n")

TENSOR A: tensor([1, 2, 3])
TENSOR B: tensor([4, 5, 6])
------------------------------------------------------------

APÓS EXECUTAR A ADIÇÃO ELEMENTO A ELEMENTO: tensor([5, 7, 9]) 



In [24]:
print("TENSOR A:", a)
print("TENSOR B:", b)
print("-" * 65)

# Multiplicação elemento a elemento (element-wise)
element_mul = a * b

print("\nAPÓS EXECUTAR A MULTIPLICAÇÃO ELEMENTO A ELEMENTO:", element_mul, "\n")

TENSOR A: tensor([1, 2, 3])
TENSOR B: tensor([4, 5, 6])
-----------------------------------------------------------------

APÓS EXECUTAR A MULTIPLICAÇÃO ELEMENTO A ELEMENTO: tensor([ 4, 10, 18]) 



* **Produto Escalar (`torch.matmul()`)**: Calcula o produto escalar (*dot product*) de dois vetores ou matrizes.

In [25]:
print("TENSOR A:", a)
print("TENSOR B:", b)
print("-" * 65)

# Produto escalar (Dot product)
dot_product = torch.matmul(a, b)

print("\nAPÓS EXECUTAR O PRODUTO ESCALAR:", dot_product, "\n")

TENSOR A: tensor([1, 2, 3])
TENSOR B: tensor([4, 5, 6])
-----------------------------------------------------------------

APÓS EXECUTAR O PRODUTO ESCALAR: tensor(32) 



* **Difusão (Broadcasting)**: A expansão automática de tensores menores para corresponder ao formato (*shape*) de tensores maiores durante operações aritméticas.
    * O broadcasting permite operações entre tensores com formatos compatíveis, mesmo que não tenham exatamente as mesmas dimensões.

In [27]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([[1],
                 [2],
                 [3]])

print("TENSOR A:", a)
print("FORMATO (SHAPE):", a.shape)
print("\nTENSOR B\n\n", b)
print("\nFORMATO (SHAPE):", b.shape)
print("-" * 65)

# Aplicar transmissão (Broadcasting)
c = a + b


print("\nTENSOR C:\n\n", c)
print("\nFORMATO (SHAPE):", c.shape, "\n")

TENSOR A: tensor([1, 2, 3])
FORMATO (SHAPE): torch.Size([3])

TENSOR B

 tensor([[1],
        [2],
        [3]])

FORMATO (SHAPE): torch.Size([3, 1])
-----------------------------------------------------------------

TENSOR C:

 tensor([[2, 3, 4],
        [3, 4, 5],
        [4, 5, 6]])

FORMATO (SHAPE): torch.Size([3, 3]) 



### 4.2 - Lógica e Comparações

As operações lógicas são ferramentas poderosas para a preparação e análise de dados. Elas permitem que você crie máscaras booleanas para filtrar, selecionar ou modificar seus dados com base em condições específicas que você definir.

* **Operadores de Comparação**: Comparações elemento a elemento (`>`, `==`, `<`) que produzem um tensor booleano.

In [28]:
temperatures = torch.tensor([20, 35, 19, 35, 42])
print("TEMPERATURAS:", temperatures)
print("-" * 50)

### Operadores de Comparação (>, <, ==)

# Use '>' (maior que) para encontrar temperaturas acima de 30
is_hot = temperatures > 30

# Use '<=' (menor ou igual a) para encontrar temperaturas iguais ou abaixo de 20
is_cool = temperatures <= 20

# Use '==' (igual a) para encontrar temperaturas exatamente iguais a 35
is_35_degrees = temperatures == 35

print("\nQUENTE (> 30 GRAUS):", is_hot)
print("FRIO (<= 20 GRAUS):", is_cool)
print("EXATAMENTE 35 GRAUS:", is_35_degrees, "\n")

TEMPERATURAS: tensor([20, 35, 19, 35, 42])
--------------------------------------------------

QUENTE (> 30 GRAUS): tensor([False,  True, False,  True,  True])
FRIO (<= 20 GRAUS): tensor([ True, False,  True, False, False])
EXATAMENTE 35 GRAUS: tensor([False,  True, False,  True, False]) 



* **Operadores Lógicos**: Operações lógicas elemento a elemento (`&` para **AND**, `|` para **OR**) em tensores booleanos.

In [29]:
is_morning = torch.tensor([True, False, False, True])
is_raining = torch.tensor([False, False, True, True])
print("É MANHÃ:", is_morning)
print("ESTÁ CHOVENDO:", is_raining)
print("-" * 50)

### Operadores Lógicos (&, |)

# Use '&' (E) para encontrar quando é manhã E está chovendo ao mesmo tempo
morning_and_raining = (is_morning & is_raining)

# Use '|' (OU) para encontrar quando é manhã OU está chovendo
morning_or_raining = is_morning | is_raining

print("\nMANHÃ & (E) CHOVENDO:", morning_and_raining)
print("MANHÃ | (OU) CHOVENDO:", morning_or_raining)

É MANHÃ: tensor([ True, False, False,  True])
ESTÁ CHOVENDO: tensor([False, False,  True,  True])
--------------------------------------------------

MANHÃ & (E) CHOVENDO: tensor([False, False, False,  True])
MANHÃ | (OU) CHOVENDO: tensor([ True, False,  True,  True])


### 4.3 - Estatísticas

Calcular estatísticas como a média ou o desvio padrão pode ser útil para entender o seu conjunto de dados ou para implementar certos tipos de normalização durante a fase de preparação dos dados.

* `torch.mean()`: Calcula a média de todos os elementos em um tensor.

In [30]:
data = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0])
print("DADOS:", data)
print("-" * 45)

# Calcular a média
data_mean = data.mean()

print("\nMÉDIA CALCULADA:", data_mean, "\n")

DADOS: tensor([10., 20., 30., 40., 50.])
---------------------------------------------

MÉDIA CALCULADA: tensor(30.) 



* `torch.std()`: Calcula o desvio padrão de todos os elementos.

In [31]:
print("DADOS:", data)
print("-" * 45)

# Calcular o desvio padrão
data_std = data.std()

print("\nDESVIO PADRÃO CALCULADO:", data_std, "\n")

DADOS: tensor([10., 20., 30., 40., 50.])
---------------------------------------------

DESVIO PADRÃO CALCULADO: tensor(15.8114) 



### 4.4 - Tipos de Dados

Tão importante quanto o formato (*shape*) de um tensor é o seu tipo de dados (*dtype*). Redes neurais geralmente realizam seus cálculos usando números de ponto flutuante de 32 bits (`float32`). Fornecer dados com o tipo incorreto, como um número inteiro, pode levar a erros de execução ou comportamentos inesperados durante o treinamento. É uma boa prática garantir que seus tensores tenham o tipo de dados correto para o seu modelo.

* **Conversão de Tipo (Type Casting - `.int()`, etc.)**: Converte um tensor de um tipo de dados para outro (ex: de ponto flutuante para inteiro).

In [32]:
print("DADOS:", data)
print("TIPO DE DADOS:", data.dtype)
print("-" * 45)

# Converter o tensor para o tipo inteiro (int)
int_tensor = data.int()

print("\nDADOS CONVERTIDOS:", int_tensor)
print("TIPO DE DADOS CONVERTIDOS:", int_tensor.dtype)

DADOS: tensor([10., 20., 30., 40., 50.])
TIPO DE DADOS: torch.float32
---------------------------------------------

DADOS CONVERTIDOS: tensor([10, 20, 30, 40, 50], dtype=torch.int32)
TIPO DE DADOS CONVERTIDOS: torch.int32


## 5 - Exercícios

Você cobriu as ferramentas essenciais para trabalhar com tensores no PyTorch. A teoria fornece o mapa, mas a prática orientada é o que constrói a verdadeira confiança e habilidade. Os seguintes exercícios opcionais são sua oportunidade de aplicar o que aprendeu em cenários práticos, desde a análise de dados de vendas até a engenharia de novas características para um modelo de aprendizado de máquina. É aqui que os conceitos realmente ganham vida, então mergulhe de cabeça e coloque seu novo conhecimento à prova!

### Exercício 1: Analisando Dados de Vendas Mensais

Você é um analista de dados em uma empresa de e-commerce. Você recebeu um tensor que representa as vendas mensais de três produtos diferentes ao longo de um período de quatro meses. Sua tarefa é extrair *insights* significativos desses dados.

O tensor `sales_data` está estruturado da seguinte forma:

* **Linhas** representam os **produtos** (Produto A, Produto B, Produto C).
* **Colunas** representam os **meses** (Jan, Fev, Mar, Abr).

**Seus objetivos são**:

1. Calcular o total de vendas para o **Produto B** (a segunda linha).
2. Identificar quais meses tiveram vendas **maiores que 130** para o **Produto C** (a terceira linha) usando máscara booleana.
3. Extrair os dados de vendas de todos os produtos para os meses de **Fev e Mar** (as duas colunas centrais).

<br>

<details>
<summary><span style="color:green;"><strong>Solução (Clique aqui para expandir)</strong></span></summary>

```python
### COMECE SEU CÓDIGO AQUI ###

# 1. Calcular o total de vendas para o Produto B.
total_sales_product_b = sales_data[1].sum()

# 2. Encontrar meses onde as vendas para o Produto C foram > 130.
high_sales_mask_product_c = sales_data[2] > 130

# 3. Obter vendas de Fev e Mar para todos os produtos.
sales_feb_mar = sales_data[:, 1:3]

### FIM DO SEU CÓDIGO AQUI ###

In [33]:
# Dados de vendas de 3 produtos ao longo de 4 meses
sales_data = torch.tensor([[100, 120, 130, 110],   # Produto A
                           [ 90,  95, 105, 125],   # Produto B
                           [140, 115, 120, 150]    # Produto C
                          ], dtype=torch.float32)

print("DADOS DE VENDAS ORIGINAIS:\n\n", sales_data)
print("-" * 45)

### COMECE SEU CÓDIGO AQUI ###

# 1. Calcular o total de vendas para o Produto B.
total_sales_product_b = None

# 2. Encontrar os meses onde as vendas do Produto C foram > 130.
high_sales_mask_product_c = None

# 3. Obter as vendas de Fev e Mar para todos os produtos.
sales_feb_mar = None

### FIM DO SEU CÓDIGO AQUI ###

print("\nTotal de Vendas do Produto B:                ", total_sales_product_b)
print("\nMeses com Vendas >130 para o Produto C (Máscara): ", high_sales_mask_product_c)
print("\nVendas de Fev & Mar:\n\n", sales_feb_mar)

DADOS DE VENDAS ORIGINAIS:

 tensor([[100., 120., 130., 110.],
        [ 90.,  95., 105., 125.],
        [140., 115., 120., 150.]])
---------------------------------------------

Total de Vendas do Produto B:                 None

Meses com Vendas >130 para o Produto C (Máscara):  None

Vendas de Fev & Mar:

 None


#### Saída Esperada:

```
Total de Vendas do Produto B:			 tensor(415.)

Meses com Vendas >130 para o Produto C (Máscara):	 tensor([ True, False, False,  True])

Vendas de Fev & Mar:

 tensor([[120., 130.],
        [ 95., 105.],
        [115., 120.]])
```

### Exercício 2: Transformação de Lote de Imagens

Você está trabalhando em um modelo de visão computacional e possui um lote de 4 imagens em tons de cinza, cada uma com um tamanho de 3 pixels de altura por 2 pixels de largura. Os dados estão atualmente em um tensor com o formato `[4, 3, 2]`, que representa `[batch_size, height, width]`.

Para o processamento com certas estruturas de aprendizado profundo, você precisa transformar esses dados para o formato `[batch_size, channels, height, width]`. Como as imagens são em tons de cinza, **você precisará**:

1. Adicionar uma nova dimensão de tamanho 1 no índice 1 para representar o canal de cor.
2. Após adicionar o canal, você percebe que o modelo espera o formato `[batch_size, height, width, channels]`. Transponha o tensor para mover a dimensão do canal para a última posição sem embaralhar a altura e a largura. Use `.transpose()` múltiplas vezes.

<br>

<details>
<summary><span style="color:green;"><strong>Solução (Clique aqui para expandir)</strong></span></summary>

```python
### COMECE SEU CÓDIGO AQUI ###

# 1. Adicionar uma dimensão de canal no índice 1.
# Resultado: [batch, channels, height, width]
image_batch_with_channel = image_batch.unsqueeze(1)

# 2. Transpor para obter [batch, height, width, channels].
# Passo A: Trocar canais (1) com altura (2)
# Resultado: [batch, height, channels, width]
temp_tensor = image_batch_with_channel.transpose(1, 2)

# Passo B: Trocar canais (2) com largura (3)
# Resultado: [batch, height, width, channels]
image_batch_transposed = temp_tensor.transpose(2, 3)

### FIM DO SEU CÓDIGO AQUI ###

In [ ]:
# Um lote (batch) de 4 imagens em tons de cinza, cada uma 3x2
image_batch = torch.rand(4, 3, 2)

print("FORMATO DO LOTE ORIGINAL:", image_batch.shape)
print("-" * 45)

### COMECE SEU CÓDIGO AQUI ###

# 1. Adicionar uma dimensão de canal no índice 1.
# Resultado esperado: [lote, canais, altura, largura]
image_batch_with_channel = None

# 2. Transpor para obter [lote, altura, largura, canais].
# Passo A: Trocar canais (índice 1) com altura (índice 2)
# Resultado: [lote, altura, canais, largura]
temp_tensor = None

# Passo B: Trocar canais (índice 2) com largura (índice 3)
# Resultado: [lote, altura, largura, canais]
image_batch_transposed = None

### FIM DO SEU CÓDIGO AQUI ###

print("\nFORMATO APÓS O UNSQUEEZE:", image_batch_with_channel.shape)
print("FORMATO APÓS A TRANSPOSIÇÃO:", image_batch_transposed.shape)

#### Saída Esperada:

```
FORMATO APÓS O UNSQUEEZE: torch.Size([4, 1, 3, 2])
FORMATO APÓS A TRANSPOSIÇÃO: torch.Size([4, 3, 2, 1])
```

### Exercício 3: Combinando e Ponderando Dados de Sensores

Você está construindo um sistema de monitoramento ambiental que utiliza dois sensores: um para temperatura e outro para umidade. Você recebe os dados desses sensores como dois tensores unidimensionais (1D) separados.

**Sua tarefa é**:

1. **Concatenar** os dois tensores em um único tensor `2x5`, onde a primeira linha são os dados de temperatura e a segunda são os de umidade.
2. Criar um tensor de pesos (`weights`) `torch.tensor([0.6, 0.4])`.
3. Usar **broadcasting e multiplicação elemento a elemento** para aplicar esses pesos aos dados combinados dos sensores. Os dados de temperatura devem ser multiplicados por 0.6 e os de umidade por 0.4.
4. Por fim, calcular a **média ponderada** para cada intervalo de tempo, **somando** os valores ponderados ao longo de `dim=0` e **dividindo** pela soma dos pesos.

<br>

<details>
<summary><span style="color:green;"><strong>Solução (Clique aqui para expandir)</strong></span></summary>

```python
### COMECE SEU CÓDIGO AQUI ###

# 1. Concatenar os dois tensores.
combined_data = torch.cat((temperature.unsqueeze(0), humidity.unsqueeze(0)), dim=0)

# 2. Criar o tensor de pesos.
weights = torch.tensor([0.6, 0.4])

# 3. Aplicar os pesos usando broadcasting.
weighted_data = combined_data * weights.unsqueeze(1)

# 4. Calcular a média ponderada para cada intervalo de tempo.
weighted_sum = torch.sum(weighted_data, dim=0)
weighted_average = weighted_sum / torch.sum(weights)

### FIM DO SEU CÓDIGO AQUI ###
```

In [ ]:
# Leituras de sensores (5 intervalos de tempo)
temperature = torch.tensor([22.5, 23.1, 21.9, 22.8, 23.5])
humidity = torch.tensor([55.2, 56.4, 54.8, 57.1, 56.8])

print("DADOS DE TEMPERATURA: ", temperature)
print("DADOS DE UMIDADE:     ", humidity)
print("-" * 45)

### COMECE SEU CÓDIGO AQUI ###

# 1. Concatenar os dois tensores.
# Nota: Você precisará usar o unsqueeze primeiro para empilhá-los verticalmente.
combined_data = None

# 2. Criar o tensor de pesos (weights).
weights = None

# 3. Aplicar os pesos usando transmissão (broadcasting).
# Você precisará remodelar (reshape) os pesos para [2, 1] para transmitir pelas colunas.
weighted_data = None

# 4. Calcular a média ponderada para cada intervalo de tempo.
#    (Média real = soma ponderada / soma dos pesos)
weighted_sum = None
weighted_average = None

### FIM DO SEU CÓDIGO AQUI ###

print("\nDADOS COMBINADOS (2x5):\n\n", combined_data)
print("\nDADOS PONDERADOS:\n\n", weighted_data)
print("\nMÉDIA PONDERADA:", weighted_average)

#### Saída Esperada:

```
COMBINADOS (2x5):

 tensor([[22.5000, 23.1000, 21.9000, 22.8000, 23.5000],
        [55.2000, 56.4000, 54.8000, 57.1000, 56.8000]])

DADOS PONDERADOS:

 tensor([[13.5000, 13.8600, 13.1400, 13.6800, 14.1000],
        [22.0800, 22.5600, 21.9200, 22.8400, 22.7200]])

MÉDIA PONDERADA: tensor([35.5800, 36.4200, 35.0600, 36.5200, 36.8200])
```

### Exercício 4: Engenharia de Atributos para Tarifas de Táxi

Você está trabalhando com um conjunto de dados de viagens de táxi. Você tem um tensor, `trip_data`, onde cada linha é uma viagem e as colunas representam **[distância (km), hora_do_dia (24h)]**.

**Seu objetivo** é criar um novo atributo binário chamado `is_rush_hour_long_trip`. Este atributo deve ser `True` (ou `1`) apenas se uma viagem atender a **ambos** os seguintes critérios:

* For uma **viagem longa** (distância > 10 km).
* Ocorrer durante um **horário de pico** (8-10h ou 17-19h).

**Passos**:
1. **Fatiar** o tensor para isolar colunas.
2. Usar **operadores lógicos** para as condições.
3. Combinar as máscaras.
4. **Redimensionar** e converter para *float*.

<br>

<details>
<summary><span style="color:green;"><strong>Solução (Clique aqui para expandir)</strong></span></summary>

```python
### COMECE SEU CÓDIGO AQUI ###

# 1. Fatiar o tensor principal.
distances = trip_data[:, 0]
hours = trip_data[:, 1]

# 2. Criar máscaras booleanas.
is_long_trip = distances > 10.0
is_morning_rush = (hours >= 8.0) & (hours < 10.0)
is_evening_rush = (hours >= 17.0) & (hours < 19.0)

# 3. Combinar as máscaras.
is_rush_hour_long_trip_mask = (is_morning_rush | is_evening_rush) & is_long_trip

# 4. Redimensionar e converter para float.
new_feature_col = is_rush_hour_long_trip_mask.float().unsqueeze(1)

### FIM DO SEU CÓDIGO AQUI ###

In [ ]:
# Dados de 8 viagens de táxi: [distância, hora_do_dia]
trip_data = torch.tensor([
    [5.3, 7],   # Fora do pico, curta
    [12.1, 9],  # Pico da manhã, longa -> PICO LONGA
    [15.5, 13], # Fora do pico, longa
    [6.7, 18],  # Pico da tarde, curta
    [2.4, 20],  # Fora do pico, curta
    [11.8, 17], # Pico da tarde, longa -> PICO LONGA
    [9.0, 9],   # Pico da manhã, curta
    [14.2, 8]   # Pico da manhã, longa -> PICO LONGA
], dtype=torch.float32)


print("DADOS ORIGINAIS DE VIAGEM (Distância, Hora):\n\n", trip_data)
print("-" * 55)


### COMECE SEU CÓDIGO AQUI ###

# 1. Fatiar o tensor principal para obter tensores 1D para cada característica.
distances = None
hours = None

# 2. Criar máscaras booleanas para cada condição.
is_long_trip = None
is_morning_rush = None
is_evening_rush = None

# 3. Combinar as máscaras para identificar viagens longas em horário de pico.
# Uma viagem é "pico longa" se for (pico da manhã OU tarde) E uma viagem longa.
is_rush_hour_long_trip_mask = None

# 4. Remodelar a nova característica em um vetor coluna e converter para float.
new_feature_col = None

### FIM DO SEU CÓDIGO AQUI ###

print("\nMÁSCARA 'VIAGEM LONGA EM HORÁRIO DE PICO': ", is_rush_hour_long_trip_mask)
print("\nCOLUNA DA NOVA CARACTERÍSTICA (Remodelada):\n\n", new_feature_col)

# Agora você pode concatenar esta nova característica aos dados originais
enhanced_trip_data = torch.cat((trip_data, new_feature_col), dim=1)
print("\nDADOS APRIMORADOS (com a nova característica ao final):\n\n", enhanced_trip_data)

#### Saída Esperada:

```
MÁSCARA 'VIAGEM LONGA EM HORÁRIO DE PICO':  tensor([False,  True, False, False, False,  True, False,  True])

COLUNA DA NOVA CARACTERÍSTICA (Reshaped):

 tensor([[0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.]])

DADOS APRIMORADOS (com a nova característica ao final):

 tensor([[ 5.3000,  7.0000,  0.0000],
        [12.1000,  9.0000,  1.0000],
        [15.5000, 13.0000,  0.0000],
        [ 6.7000, 18.0000,  0.0000],
        [ 2.4000, 20.0000,  0.0000],
        [11.8000, 17.0000,  1.0000],
        [ 9.0000,  9.0000,  0.0000],
        [14.2000,  8.0000,  1.0000]])
```        

## Conclusão

Você concluiu este .aboratório! Trabalhou com os blocos de construção fundamentais do PyTorch. Começou do zero e aprendeu a criar, redimensionar, combinar e consultar tensores de diversas maneiras.

As habilidades que você desenvolveu aqui são essenciais para todo profissional de aprendizado de máquina. A aritmética elemento a elemento e o *broadcasting* que você praticou são exatamente a forma como uma rede neural aplica pesos e vieses a lotes inteiros de dados de uma só vez com eficiência. As técnicas de redimensionamento como `unsqueeze` e `squeeze` são o que permitem preparar um único ponto de dado para um modelo que espera um lote e, depois, limpar a saída. Estes não são apenas exercícios abstratos; são as operações do dia a dia necessárias para construir e depurar modelos de *deep learning* eficazes.

Com esta compreensão sólida de tensores, você está agora totalmente preparado para seguir para a próxima etapa: construir e treinar redes neurais para resolver problemas ainda mais complexos. Cada modelo que você construir de agora em diante se apoiará nesta base.